# 01 Data Cleaning and Ingredient Analysis

This notebook:
- Filters dataset to skincare
- Cleans ingredient lists
- Analyzes ingredient frequency distribution
- Selects statistically meaningful ingredient subset (≥50 occurrences)

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import ast
from collections import Counter


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
PRODUCT_PATH = "/content/drive/MyDrive/skincare-ai/data/raw/product_info.csv"

products = pd.read_csv(PRODUCT_PATH)
products.head()

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0


# Filter Skin Care Products

In [7]:
skincare = products[products['primary_category'] == 'Skincare'].copy()

print("Total products:", len(products))
print("Skincare products:", len(skincare))

Total products: 8494
Skincare products: 2420


In [9]:
skincare['ingredients'].head()

,ingredients
89,"['Collagen (Vegan)*, Water (Aqua, Eau), Ethylh..."
90,"['Collagen (Vegan)*, Water (Aqua, Eau), Propan..."
91,"['Aqua (Water/Eau), Stearic Acid, Isopropyl Is..."
92,"['Collagen (Vegan)*, Water (Aqua, Eau), Glycer..."
93,"['Octinoxate 7.5%, Titanium Dioxide 2%, Zinc O..."


In [11]:
import re

def clean_ingredients(ingredient_string):
    if pd.isna(ingredient_string):
        return []

    try:
        # Convert string representation to actual Python list
        parsed = ast.literal_eval(ingredient_string)

        if isinstance(parsed, list) and len(parsed) > 0:
            ingredient_text = parsed[0]
        else:
            return []

        # Split on commas NOT inside parentheses
        ingredients = re.split(r',(?![^()]*\))', ingredient_text)

        # Clean formatting
        ingredients = [ing.strip().lower() for ing in ingredients if ing.strip() != ""]

        return ingredients

    except:
        return []

In [12]:
skincare['ingredient_list'] = skincare['ingredients'].apply(clean_ingredients)

<unknown>:1: SyntaxWarning: invalid escape sequence '\A'
<unknown>:1: SyntaxWarning: invalid escape sequence '\A'


In [14]:
skincare[['ingredients', 'ingredient_list']].head(3)

,ingredients,ingredient_list
89,"['Collagen (Vegan)*, Water (Aqua, Eau), Ethylh...","[collagen (vegan)*, water (aqua, eau), ethylhe..."
90,"['Collagen (Vegan)*, Water (Aqua, Eau), Propan...","[collagen (vegan)*, water (aqua, eau), propane..."
91,"['Aqua (Water/Eau), Stearic Acid, Isopropyl Is...","[aqua (water/eau), stearic acid, isopropyl iso..."


# Frequency Analysis

In [18]:
all_ingredients = skincare['ingredient_list'].explode()

ingredient_counts = Counter(all_ingredients)

ingredient_freq = pd.DataFrame(
    ingredient_counts.items(),
    columns=['ingredient', 'count']
).sort_values(by='count', ascending=False)

ingredient_freq.head(20)

,ingredient,count
5,glycerin,1542
33,phenoxyethanol,930
44,butylene glycol,923
202,sodium hyaluronate,729
20,tocopherol,724
27,citric acid,722
42,propanediol,706
152,xanthan gum,669
25,ethylhexylglycerin,647
23,caprylyl glycol,628


In [19]:
print("Total unique ingredients:", ingredient_freq.shape[0])

print("Ingredients appearing ≥ 10 times:",
      ingredient_freq[ingredient_freq['count'] >= 10].shape[0])

print("Ingredients appearing ≥ 25 times:",
      ingredient_freq[ingredient_freq['count'] >= 25].shape[0])

print("Ingredients appearing ≥ 50 times:",
      ingredient_freq[ingredient_freq['count'] >= 50].shape[0])

Total unique ingredients: 7264
Ingredients appearing ≥ 10 times: 1046
Ingredients appearing ≥ 25 times: 473
Ingredients appearing ≥ 50 times: 224


In [20]:
top_ingredients = set(
    ingredient_freq[ingredient_freq['count'] >= 50]['ingredient']
)

# Top Ingredient Selection

In [21]:
def encode_ingredients(ingredient_list):
    return [
        1 if ing in ingredient_list else 0
        for ing in top_ingredients
    ]

X_ingredients = skincare['ingredient_list'].apply(encode_ingredients)
X_ingredients = np.array(X_ingredients.tolist())

y = skincare['rating'].values

print("Feature matrix shape:", X_ingredients.shape)
print("Target shape:", y.shape)

Feature matrix shape: (2420, 224)
Target shape: (2420,)
